# 🛣️ RoadScan AI — Pothole Detection (YOLOv8m Training)
**Thesis: Pothole Detection Web App for LGUs**

This notebook trains a YOLOv8m model on the **RDD2022 dataset (D40 pothole class only)**.

### What this notebook does:
1. ⬇️ Downloads the RDD2022 dataset **directly in Colab** (no manual upload)
2. 📦 Extracts nested country zip files
3. 🔍 Filters for **pothole (D40) annotations only**
4. 🔄 Converts VOC XML → YOLO format
5. 🏋️ Trains **YOLOv8m** at **1024×1024** resolution for **150 epochs**
6. 📊 Evaluates results (target: **mAP@50 ≥ 80%**)
7. 💾 Downloads `best.pt` for the web app

### ⚡ Before running:
- Go to **Runtime → Change runtime type → T4 GPU** (or better)
- Estimated training time: **3-5 hours** on T4, **~1 hour** on A100

---
## 1. Setup & GPU Check

In [ ]:
# Install dependencies
!pip install ultralytics -q

import torch
import os
import shutil

print("=" * 50)
print("🔧 ENVIRONMENT CHECK")
print("=" * 50)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
else:
    print("⚠️  NO GPU DETECTED!")
    print("Go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required for training")

# Check disk space
disk = shutil.disk_usage('/')
print(f"Disk space: {disk.free / 1e9:.1f} GB free / {disk.total / 1e9:.1f} GB total")
print("=" * 50)

---
## 2. Download RDD2022 Dataset
Downloads the full RDD2022 dataset (~13GB) directly from Figshare.

**Skip this cell if you already downloaded it.**

In [ ]:
import os
import time

DOWNLOAD_URL = "https://ndownloader.figshare.com/files/38030910"
ZIP_PATH = "/content/RDD2022.zip"
EXTRACT_DIR = "/content/RDD2022_raw"

# Skip download if already extracted
if os.path.exists(EXTRACT_DIR) and len(os.listdir(EXTRACT_DIR)) > 0:
    print("✅ RDD2022 already downloaded and extracted. Skipping.")
else:
    # Download
    print("⬇️  Downloading RDD2022 from Figshare (~13GB)...")
    print("   This takes ~5-10 minutes. Go grab a coffee ☕")
    start = time.time()
    !wget -q --show-progress -O {ZIP_PATH} {DOWNLOAD_URL}
    elapsed = time.time() - start
    print(f"\n✅ Download complete in {elapsed/60:.1f} minutes")

    # Verify file size
    size_gb = os.path.getsize(ZIP_PATH) / 1e9
    print(f"   File size: {size_gb:.2f} GB")

    # Extract
    print("\n📦 Extracting dataset...")
    !mkdir -p {EXTRACT_DIR}
    !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
    print("✅ Extraction complete!")

    # Remove zip to free disk space
    os.remove(ZIP_PATH)
    print("🗑️  Removed zip file to free disk space")

# Show what we got
print("\n📁 Dataset contents:")
!find {EXTRACT_DIR} -maxdepth 2 -type d | head -30

---
## 2b. Extract Nested Country Zip Files
The RDD2022 main zip contains **per-country zip files** (e.g. `China_Drone.zip`, `Japan.zip`). This cell extracts them.

**Run this even if you already downloaded — it handles the nested zips.**

In [ ]:
import os
import glob

EXTRACT_DIR = "/content/RDD2022_raw"

# Find all nested zip files
nested_zips = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith('.zip'):
            nested_zips.append(os.path.join(root, f))

if nested_zips:
    print(f"📦 Found {len(nested_zips)} nested zip files to extract:")
    for zf in nested_zips:
        zf_name = os.path.basename(zf)
        zf_dir = os.path.dirname(zf)
        print(f"   📦 Extracting {zf_name}...")
        !unzip -q -o "{zf}" -d "{zf_dir}"
        os.remove(zf)
        print(f"      ✅ Done, removed {zf_name}")
    print(f"\n✅ All {len(nested_zips)} country zips extracted!")
else:
    print("✅ No nested zips found — country folders already extracted.")

# Show final structure
print("\n📁 Final dataset structure:")
!find {EXTRACT_DIR} -maxdepth 4 -type d | sort | head -40

# Check disk space
import shutil
disk = shutil.disk_usage('/')
print(f"\n💾 Disk space remaining: {disk.free / 1e9:.1f} GB")

---
## 3. Filter for Pothole (D40) & Convert to YOLO Format
The RDD2022 has 4 damage classes: D00 (longitudinal crack), D10 (transverse crack), D20 (alligator crack), **D40 (pothole)**.

We filter for **D40 only** since this is a pothole detection app.

In [ ]:
import xml.etree.ElementTree as ET
import glob
import random
import shutil
from pathlib import Path

# --- Configuration ---
RAW_DIR = "/content/RDD2022_raw"  # Where we extracted the dataset
OUTPUT_DIR = "/content/pothole_dataset"  # YOLO-formatted output
TARGET_CLASS = "D40"  # Pothole class in RDD2022
RANDOM_SEED = 42

# Train/Val/Test split ratios
TRAIN_RATIO = 0.80
VAL_RATIO = 0.15
# TEST_RATIO = 0.05 (remainder)

# Countries in RDD2022
COUNTRIES = ["China_Drone", "China_MotorBike", "Czech", "India", "Japan", "Norway", "United_States"]


def parse_voc_xml(xml_path, target_class="D40"):
    """Parse a VOC XML annotation file and return YOLO-format labels for the target class."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find('size')
        if size is None:
            return []

        img_w = int(size.find('width').text)
        img_h = int(size.find('height').text)

        if img_w == 0 or img_h == 0:
            return []

        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text.strip()

            if name != target_class:
                continue

            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)

            # Convert to YOLO format (normalized center_x, center_y, width, height)
            cx = ((xmin + xmax) / 2.0) / img_w
            cy = ((ymin + ymax) / 2.0) / img_h
            w = (xmax - xmin) / img_w
            h = (ymax - ymin) / img_h

            # Clamp to [0, 1]
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w = max(0.001, min(1.0, w))
            h = max(0.001, min(1.0, h))

            # Class 0 = pothole (single-class model)
            labels.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        return labels
    except Exception as e:
        return []


# --- Collect all images with D40 (pothole) annotations ---
print("=" * 60)
print("🔍 FILTERING RDD2022 FOR POTHOLE (D40) CLASS")
print("=" * 60)

all_pairs = []  # List of (image_path, [yolo_labels])
stats = {}  # Per-country stats

# Find the actual RDD2022 root (may be nested)
rdd_root = None
for root_dir, dirs, files in os.walk(RAW_DIR):
    if any(c in dirs for c in COUNTRIES):
        rdd_root = root_dir
        break

if rdd_root is None:
    # Try common nested paths
    possible = [
        os.path.join(RAW_DIR, "RDD2022"),
        os.path.join(RAW_DIR, "RDD2022_released_through_CRDDC2022"),
        RAW_DIR,
    ]
    for p in possible:
        if os.path.exists(p):
            rdd_root = p
            break

print(f"📂 RDD2022 root: {rdd_root}")
print()

for country in COUNTRIES:
    country_dir = os.path.join(rdd_root, country)
    if not os.path.exists(country_dir):
        print(f"   ⚠️ {country}: not found, skipping")
        continue

    country_count = 0
    country_bboxes = 0

    # Process train split (has annotations)
    for split in ['train']:
        img_dir = os.path.join(country_dir, split, 'images')
        xml_dir = os.path.join(country_dir, split, 'annotations', 'xmls')

        if not os.path.exists(xml_dir):
            xml_dir = os.path.join(country_dir, split, 'annotations')

        if not os.path.exists(img_dir) or not os.path.exists(xml_dir):
            continue

        for xml_file in glob.glob(os.path.join(xml_dir, '*.xml')):
            labels = parse_voc_xml(xml_file, TARGET_CLASS)

            if labels:
                # Find corresponding image
                img_stem = os.path.splitext(os.path.basename(xml_file))[0]
                img_path = None

                for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                    candidate = os.path.join(img_dir, img_stem + ext)
                    if os.path.exists(candidate):
                        img_path = candidate
                        break

                if img_path:
                    all_pairs.append((img_path, labels))
                    country_count += 1
                    country_bboxes += len(labels)

    stats[country] = {'images': country_count, 'bboxes': country_bboxes}
    print(f"   🌍 {country}: {country_count} images, {country_bboxes} pothole bboxes")

total_images = len(all_pairs)
total_bboxes = sum(len(labels) for _, labels in all_pairs)
print()
print(f"📊 TOTAL: {total_images} images with {total_bboxes} pothole bounding boxes")

if total_images == 0:
    raise RuntimeError("No pothole (D40) annotations found! Check the dataset path.")

# --- Shuffle and split ---
random.seed(RANDOM_SEED)
random.shuffle(all_pairs)

n = len(all_pairs)
train_end = int(TRAIN_RATIO * n)
val_end = int((TRAIN_RATIO + VAL_RATIO) * n)

splits = {
    'train': all_pairs[:train_end],
    'val': all_pairs[train_end:val_end],
    'test': all_pairs[val_end:],
}

# --- Create YOLO dataset ---
print()
print("📁 Creating YOLO-formatted dataset...")

# Clean output directory
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split_name, pairs in splits.items():
    img_out = os.path.join(OUTPUT_DIR, split_name, 'images')
    lbl_out = os.path.join(OUTPUT_DIR, split_name, 'labels')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for idx, (img_path, labels) in enumerate(pairs):
        ext = os.path.splitext(img_path)[1]
        new_name = f"pothole_{idx:05d}"

        # Copy image
        shutil.copy2(img_path, os.path.join(img_out, new_name + ext))

        # Write YOLO label file
        with open(os.path.join(lbl_out, new_name + '.txt'), 'w') as f:
            f.write('\n'.join(labels))

    split_bboxes = sum(len(l) for _, l in pairs)
    print(f"   {split_name}: {len(pairs)} images, {split_bboxes} bboxes")

# --- Create data.yaml ---
yaml_content = f"""# Pothole Detection Dataset
# Source: RDD2022 (D40 class only)
# Total: {total_images} images, {total_bboxes} pothole annotations

path: {OUTPUT_DIR}
train: train/images
val: val/images
test: test/images

nc: 1

names:
  0: pothole
"""

yaml_path = os.path.join(OUTPUT_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ Dataset ready at: {OUTPUT_DIR}")
print(f"📝 data.yaml: {yaml_path}")

---
## 4. Verify Dataset (Visual Check)
Let's visually verify that the annotations are correct before training.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

def show_yolo_sample(dataset_dir, split='train', num_samples=6):
    """Display random samples with bounding boxes overlaid."""
    img_dir = os.path.join(dataset_dir, split, 'images')
    lbl_dir = os.path.join(dataset_dir, split, 'labels')

    images = sorted(os.listdir(img_dir))
    samples = random.sample(images, min(num_samples, len(images)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Sample {split} images with pothole annotations', fontsize=16, fontweight='bold')

    for idx, (ax, img_name) in enumerate(zip(axes.flat, samples)):
        img = cv2.imread(os.path.join(img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # Read labels
        lbl_name = os.path.splitext(img_name)[0] + '.txt'
        lbl_path = os.path.join(lbl_dir, lbl_name)

        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        x2 = int((cx + bw/2) * w)
                        y2 = int((cy + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
                        cv2.putText(img, 'pothole', (x1, y1-10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        ax.imshow(img)
        ax.set_title(img_name, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_yolo_sample(OUTPUT_DIR, split='train', num_samples=6)
print("\n👆 Verify: Red boxes should be around potholes. If they look wrong, check the conversion.")

---
## 5. Train YOLOv8m 🏋️

**Key improvements over previous attempts:**

| Setting | Previous (50% mAP) | Current (target 80%+) |
|---------|--------------------|-----------------------|
| Model | YOLOv8n (3.2M params) | **YOLOv8m** (25.9M params) |
| Image size | 640 | **1024** |
| Epochs | 50 | **150** |
| Batch size | 16 | **8** (fits 1024px in T4) |
| Patience | 20 | **30** |
| Cosine LR | ❌ | ✅ |
| Close mosaic | ❌ | **Last 20 epochs** |
| Label smoothing | ❌ | **0.1** |

In [ ]:
from ultralytics import YOLO
import time

# Load pretrained YOLOv8m
model = YOLO('yolov8m.pt')

print("=" * 60)
print("🏋️ STARTING TRAINING — YOLOv8m POTHOLE DETECTION")
print("=" * 60)
print(f"Model: YOLOv8m (25.9M parameters)")
print(f"Image size: 1024×1024")
print(f"Max epochs: 150 (early stopping patience: 30)")
print(f"Estimated time: 3-5 hours on T4, ~1 hour on A100")
print("=" * 60)

start_time = time.time()

results = model.train(
    data='/content/pothole_dataset/data.yaml',

    # --- Core settings ---
    epochs=150,
    batch=8,                  # Reduced for 1024px on T4 (15GB VRAM)
    imgsz=1024,               # Higher res = better small object detection
    project='pothole_training',
    name='yolov8m_1024',
    exist_ok=True,

    # --- Optimizer ---
    optimizer='auto',         # Let Ultralytics choose (SGD w/ momentum)
    lr0=0.01,                 # Default for YOLOv8
    lrf=0.1,                  # Final LR = lr0 * 0.1 (less aggressive decay)
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,          # More warmup for stability
    warmup_momentum=0.8,
    cos_lr=True,              # Cosine annealing schedule

    # --- Data augmentation ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,             # More rotation
    translate=0.2,            # More translation
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,           # Copy-paste augmentation
    close_mosaic=20,          # Disable mosaic last 20 epochs

    # --- Regularization ---
    label_smoothing=0.1,      # Prevent overconfident predictions

    # --- Training control ---
    patience=30,              # More runway before early stopping
    save=True,
    save_period=25,           # Save checkpoint every 25 epochs
    val=True,
    plots=True,
    verbose=True,
)

elapsed = time.time() - start_time
print(f"\n⏱️ Training completed in {elapsed/3600:.1f} hours")

---
## 6. Evaluate Results 📊

In [ ]:
# Run validation on the best model
best_model_path = 'pothole_training/yolov8m_1024/weights/best.pt'

if not os.path.exists(best_model_path):
    for p in [
        'runs/detect/yolov8m_1024/weights/best.pt',
        'pothole_training/yolov8m_10242/weights/best.pt',
    ]:
        if os.path.exists(p):
            best_model_path = p
            break

print(f"📦 Loading best model from: {best_model_path}")
best_model = YOLO(best_model_path)

# Validate
val_results = best_model.val(
    data='/content/pothole_dataset/data.yaml',
    imgsz=1024,
    batch=8,
    split='test',
)

print()
print("=" * 60)
print("📊 FINAL TEST RESULTS")
print("=" * 60)
print(f"   mAP@50:        {val_results.box.map50:.4f}  ({val_results.box.map50:.1%})")
print(f"   mAP@50-95:     {val_results.box.map:.4f}  ({val_results.box.map:.1%})")
print(f"   Precision:     {val_results.box.mp:.4f}  ({val_results.box.mp:.1%})")
print(f"   Recall:        {val_results.box.mr:.4f}  ({val_results.box.mr:.1%})")
print("=" * 60)

if val_results.box.map50 >= 0.80:
    print("\n🎉🎉🎉 TARGET ACHIEVED: mAP@50 ≥ 80%! 🎉🎉🎉")
    print("Your model is ready for the web app!")
elif val_results.box.map50 >= 0.70:
    print(f"\n⚠️ mAP@50 = {val_results.box.map50:.1%} — Close! Try:")
    print("  1. Train for more epochs (increase to 200-300)")
    print("  2. Use YOLOv8l (large) if you have Colab Pro")
    print("  3. The model may still be usable for demo purposes")
else:
    print(f"\n❌ mAP@50 = {val_results.box.map50:.1%} — Below target. Try:")
    print("  1. Check annotations in Step 4 — are boxes correct?")
    print("  2. Increase epochs to 200")
    print("  3. Try YOLOv8l with --imgsz 1280")

### Training Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display

train_dir = 'pothole_training/yolov8m_1024'

results_img = os.path.join(train_dir, 'results.png')
if os.path.exists(results_img):
    print("📈 Training Curves:")
    display(Image(filename=results_img, width=900))
else:
    print("⚠️ results.png not found")

cm_img = os.path.join(train_dir, 'confusion_matrix_normalized.png')
if os.path.exists(cm_img):
    print("\n📊 Confusion Matrix:")
    display(Image(filename=cm_img, width=600))

f1_img = os.path.join(train_dir, 'F1_curve.png')
if os.path.exists(f1_img):
    print("\n📊 F1 Curve:")
    display(Image(filename=f1_img, width=600))

pr_img = os.path.join(train_dir, 'PR_curve.png')
if os.path.exists(pr_img):
    print("\n📊 Precision-Recall Curve:")
    display(Image(filename=pr_img, width=600))

### Sample Predictions on Validation Images

In [ ]:
train_dir = 'pothole_training/yolov8m_1024'

val_pred = os.path.join(train_dir, 'val_batch0_pred.jpg')
val_labels = os.path.join(train_dir, 'val_batch0_labels.jpg')

if os.path.exists(val_labels):
    print("📸 Ground Truth (actual labels):")
    display(Image(filename=val_labels, width=900))

if os.path.exists(val_pred):
    print("\n📸 Model Predictions:")
    display(Image(filename=val_pred, width=900))
    print("\n👆 Compare the two images above — predictions should match ground truth.")

---
## 7. Test on Sample Images
Run inference on random test images to see the model in action.

In [ ]:
import cv2
import matplotlib.pyplot as plt

test_img_dir = '/content/pothole_dataset/test/images'
test_images = sorted(os.listdir(test_img_dir))
samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('🔍 Model Predictions on Test Images', fontsize=16, fontweight='bold')

for ax, img_name in zip(axes.flat, samples):
    img_path = os.path.join(test_img_dir, img_name)

    results = best_model.predict(img_path, imgsz=1024, conf=0.25, verbose=False)

    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    num_detections = len(results[0].boxes)
    ax.imshow(annotated)
    ax.set_title(f'{img_name} — {num_detections} detection(s)', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 8. Download Trained Model 💾

Download `best.pt` and place it in your project:
```
thesisproto/ml/models/best.pt
```
The web app backend will automatically use it for inference.

In [ ]:
from google.colab import files
import shutil

best_pt = 'pothole_training/yolov8m_1024/weights/best.pt'

if not os.path.exists(best_pt):
    for p in [
        'runs/detect/yolov8m_1024/weights/best.pt',
        'pothole_training/yolov8m_10242/weights/best.pt',
    ]:
        if os.path.exists(p):
            best_pt = p
            break

if os.path.exists(best_pt):
    size_mb = os.path.getsize(best_pt) / 1e6
    print(f"📦 Best model: {best_pt}")
    print(f"   Size: {size_mb:.1f} MB")
    print()

    output_name = 'best.pt'
    shutil.copy2(best_pt, output_name)

    print("⬇️  Downloading best.pt...")
    print("   After download, place it at: thesisproto/ml/models/best.pt")
    files.download(output_name)
else:
    print("❌ best.pt not found!")
    print("   Check the training output directory for weights.")

---
## 🔧 Troubleshooting

### If mAP is still below 80%:

**Option A: Train longer**
```python
# Re-run Step 5 with more epochs
model = YOLO('pothole_training/yolov8m_1024/weights/last.pt')
model.train(data='/content/pothole_dataset/data.yaml', epochs=100, resume=True)
```

**Option B: Use a larger model (requires Colab Pro / A100)**
```python
model = YOLO('yolov8l.pt')  # Large model (43.7M params)
model.train(data='/content/pothole_dataset/data.yaml', epochs=150,
            imgsz=1024, batch=4)  # Smaller batch for larger model
```

**Option C: Higher resolution**
```python
model = YOLO('yolov8m.pt')
model.train(data='/content/pothole_dataset/data.yaml', epochs=150,
            imgsz=1280, batch=4)  # Even higher resolution
```